In [ ]:
# Example 1: Validate a minimal valid Swagger 2.0 spec.
from swagger_spec_validator.validator20 import validate_spec

spec = {
    "swagger": "2.0",
    "info": {"title": "Test API", "version": "1.0.0"},
    "paths": {
        "/ping": {
            "get": {
                "responses": {
                    "200": {"description": "OK"}
                }
            }
        }
    },
}

result = validate_spec(spec)
print(f"validate_spec returned: {type(result).__name__}")
print("Swagger 2.0 spec validated successfully")

In [ ]:
# Example 2: Detect an invalid Swagger 2.0 spec (missing required 'info' field).
from swagger_spec_validator.validator20 import validate_spec
from swagger_spec_validator.common import SwaggerValidationError

bad_spec = {
    "swagger": "2.0",
    # missing required 'info' and 'paths'
}

raised = False
try:
    validate_spec(bad_spec)
except SwaggerValidationError as e:
    raised = True
    print(f"Caught SwaggerValidationError: {str(e)[:120]}")
except Exception as e:
    raised = True
    print(f"Caught {type(e).__name__}: {str(e)[:120]}")

assert raised, "Expected validation error for malformed spec"

In [ ]:
# Example 3: $ref resolution — validator20 resolves refs into definitions and
# rejects specs whose $ref targets don't exist.
from swagger_spec_validator.validator20 import validate_spec
from swagger_spec_validator.common import SwaggerValidationError

spec_with_ref = {
    "swagger": "2.0",
    "info": {"title": "Ref API", "version": "1.0.0"},
    "definitions": {
        "Pet": {"type": "object", "properties": {"name": {"type": "string"}}}
    },
    "paths": {
        "/pet": {
            "get": {
                "responses": {
                    "200": {"description": "OK", "schema": {"$ref": "#/definitions/Pet"}}
                }
            }
        }
    },
}

validate_spec(spec_with_ref)
print("spec with $ref into definitions validated successfully")

import copy
broken = copy.deepcopy(spec_with_ref)
broken["paths"]["/pet"]["get"]["responses"]["200"]["schema"]["$ref"] = "#/definitions/Missing"
try:
    validate_spec(broken)
    raised = False
except SwaggerValidationError as e:
    raised = True
    print(f"broken $ref correctly rejected: {str(e)[:100]}")
assert raised, "expected SwaggerValidationError for dangling $ref"

In [ ]:
# Example 4: Generic validator dispatcher — auto-detects Swagger 1.2 vs 2.0,
# and (by design) rejects OpenAPI 3.x: this package validates Swagger 1.2/2.0 only.
from swagger_spec_validator.util import get_validator
from swagger_spec_validator.common import SwaggerValidationError

spec_v2 = {
    "swagger": "2.0",
    "info": {"title": "V2", "version": "1.0"},
    "paths": {},
}
spec_v12 = {
    "swaggerVersion": "1.2",
    "basePath": "http://localhost/api",
    "apis": [],
}

v2_validator = get_validator(spec_v2, 'memory://spec_v2')
v12_validator = get_validator(spec_v12, 'memory://spec_v12')
print(f"2.0 dispatched to: {v2_validator.__name__}")
print(f"1.2 dispatched to: {v12_validator.__name__}")
assert v2_validator.__name__.endswith('validator20')
assert v12_validator.__name__.endswith('validator12')

try:
    get_validator({"openapi": "3.0.0", "info": {}, "paths": {}}, 'memory://spec_v3')
    raised = False
except SwaggerValidationError as e:
    raised = True
    print(f"OpenAPI 3.x correctly unsupported: {str(e)[:80]}")
assert raised, "expected SwaggerValidationError for OpenAPI 3.x spec"

In [ ]:
# Example 5: Validate a spec loaded from a temporary YAML file via file:// URL.
import tempfile
import os
import yaml
from swagger_spec_validator.validator20 import validate_spec_url

spec = {
    "swagger": "2.0",
    "info": {"title": "From file", "version": "1.0.0"},
    "paths": {
        "/status": {
            "get": {"responses": {"200": {"description": "OK"}}}
        }
    },
}

with tempfile.NamedTemporaryFile('w', suffix='.yaml', delete=False) as f:
    yaml.safe_dump(spec, f)
    path = f.name

try:
    result = validate_spec_url(f"file://{path}")
    print(f"validate_spec_url returned: {type(result).__name__}")
    assert result is not None or result is None  # any non-raising return is success
    print("File-based spec validation passed")
finally:
    os.unlink(path)